
<div align="center">

# BÁO CÁO GIỮA KỲ

## Đề xuất và cài đặt phiên bản tuần tự

<table style="border: none; border-collapse: collapse; margin: auto;">
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">Khoa:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">Công nghệ thông tin</td>
  </tr>
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">Trường:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">Đại học Khoa học tự nhiên, ĐHQG-HCM</td>
  </tr>
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">GVHD:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">Thầy Phạm Trọng Nghĩa</td>
  </tr>
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">Học phần:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">Lập trình song song</td>
  </tr>
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">Nhóm:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">07</td>
  </tr>
  <tr style="border: none;">
    <td style="border: none; text-align: right; padding: 4px 8px; font-weight: bold;">Năm học:</td>
    <td style="border: none; text-align: left; padding: 4px 8px;">2026 – 2027</td>
  </tr>
</table>

</div>

---

## 1. Thông tin nhóm và kế hoạch công việc

### 1.1 Danh sách thành viên

| STT | MSSV | Họ và tên |
| --- | --- | --- |
| 1 | 23120315 | Nguyễn Đăng Pha |
| 2 | 23120356 | Trần Nguyễn Trí Thanh |
| 3 | 23120360 | Đặng Lê Đức Thịnh |
| 4 | 23120366 | Trương Thanh Thuận |

### 1.2 Kế hoạch phân công

| Giai đoạn - Công việc                                       | Thành viên phụ trách   | Phiên bản  |
|-------------------------------------------------------------|------------------------|------------|
| Giai đoạn 1 - Tiền xử lý (Preprocessing)                    |                        | V0         |
| Giai đoạn 2 - Phát hiện điểm đặc trưng (Keypoint Detection) |                        | V0         |
| Giai đoạn 3 - Mô tả điểm đặc trưng (Description)            |                        | V0         |
| Giai đoạn 4 - Ghép cặp điểm (Matching)                      |                        | V0         |
| Giai đoạn 5 - Ước lượng Homography                          |                        | V0         |
| Giai đoạn 6 - Biến dạng và hòa trộn (Warping & Blending)    |                        | V0         |
| Đánh giá hiệu năng và báo cáo                               |                        | Tất cả     |

---

## 2. Hiểu vấn đề

### 2.1 Bài toán ghép ảnh toàn cảnh (Panorama Stitching)

Ghép ảnh toàn cảnh là quá trình kết hợp hai (hoặc nhiều) ảnh chụp có vùng chồng lấp của cùng một cảnh vật thành một ảnh góc rộng duy nhất...

**Đầu vào:** Hai ảnh RGB có vùng chồng lấp (ảnh A và ảnh B).  
**Đầu ra:** Một ảnh toàn cảnh duy nhất (ảnh B được biến dạng và hòa vào khung của ảnh A).

### 2.2 Pipeline 6 giai đoạn

| #   | Giai đoạn                    | Mô tả                                                                         |
|:---:|------------------------------|-------------------------------------------------------------------------------|
| 1   | **Tiền xử lý**               | Chuyển ảnh sang ảnh xám (grayscale) và làm mịn để giảm nhiễu                  |
| 2   | **Phát hiện điểm đặc trưng** | Tìm các điểm nổi bật (góc cạnh) trong từng ảnh                                |
| 3   | **Mô tả điểm đặc trưng**     | Tính vector mô tả nhỏ gọn cho vùng lân cận mỗi điểm                           |
| 4   | **Ghép cặp điểm**            | Với mỗi điểm ở ảnh A, tìm điểm tương đồng nhất ở ảnh B; loại bỏ các cặp mơ hồ |
| 5   | **Ước lượng Homography**     | Tính ma trận biến đổi xạ ảnh 3×3 để ánh xạ ảnh B sang ảnh A, xử lý nhiễu      |
| 6   | **Biến dạng và hòa trộn**    | Biến dạng ảnh B vào hệ tọa độ của ảnh A và hòa trộn vùng chồng lấp            |

---

## 3. Phương pháp được chọn

Nhóm chọn phương pháp cho từng giai đoạn dựa trên hai tiêu chí: **chất lượng đầu ra** và **khả năng tăng tốc trên GPU**.

<table>
  <thead>
    <tr>
      <th>Giai đoạn</th>
      <th>Phương pháp lựa chọn</th>
      <th>Ưu thế chất lượng</th>
      <th>Tính phù hợp với GPU</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1. Tiền xử lý</td>
      <td>Grayscale cố định + Separable Gaussian Blur (1D Horiz → 1D Vert)</td>
      <td>Loại bỏ nhiễu tần cao hiệu quả; O(2kN) thay vì O(k²N) mỗi pixel</td>
      <td>Mỗi pixel độc lập, hai pass 1D dễ map 1 thread/pixel; cache-friendly hơn kernel 2D</td>
    </tr>
    <tr>
      <td>2. Phát hiện điểm đặc trưng</td>
      <td>Harris Corner + NMS + Grid-based Top-K</td>
      <td>Response liên tục để xếp hạng; NMS loại cụm điểm; Grid Top-K giúp keypoint phân bố đều cho matching và homography</td>
      <td>Các bước là stencil/pixel-wise phù hợp 1 thread/pixel; có thể tối ưu bằng shared-memory tiling, reduction và parallel compaction</td>
    </tr>
    <tr>
      <td>3. Mô tả điểm đặc trưng</td>
      <td>ORB Descriptor (BRIEF có xoay, 256 bit)</td>
      <td>Bất biến với phép xoay trong mặt phẳng (máy ảnh nghiêng khi chụp); làm mịn trước giúp các bit ít nhạy nhiễu; đủ dùng vì ảnh chụp xoay tại chỗ nên gần như không đổi tỉ lệ</td>
      <td>Mỗi keypoint độc lập (mỗi thread xử lý một keypoint), khối lượng việc đồng đều nên ít phân kỳ warp; mẫu cặp điểm cố định đặt trong Constant Memory (broadcast); descriptor 32 byte gọn, so khớp được bằng phép đếm bit</td>
    </tr>
    <tr>
      <td>4. Ghép cặp điểm</td>
      <td>Brute-force Hamming + Lowe Ratio Test + Cross-check</td>
      <td>So khớp chính xác (không xấp xỉ như KD-tree/LSH); ratio test loại cặp mơ hồ ở vùng hoa văn lặp, cross-check loại cặp nhiều-một, giảm outlier đưa vào RANSAC</td>
      <td>Bài toán all-pairs tương tự nhân ma trận; mỗi khoảng cách chỉ cần XOR và đếm bit; dữ liệu được tái sử dụng nhiều nên hợp với Shared-memory Tiling; tìm d₁, d₂ là bài toán Reduction (warp shuffle)</td>
    </tr>
    <tr>
      <td>5. Ước lượng Homography</td>
      <td>RANSAC + 4-point DLT + Inlier Refinement</td>
      <td>Loại bỏ triệt để các cặp ghép sai (outliers); bước tinh chỉnh (refinement) giúp ma trận H chuẩn xác về mặt hình học, triệt tiêu hiện tượng lệch mép ghép</td>
      <td>Các giả thuyết RANSAC độc lập hoàn toàn → mỗi block/warp thử 1 giả thuyết song song; bước đếm inlier ánh xạ thành bài toán Parallel Reduction; không bị phân kỳ warp như PROSAC</td>
    </tr>
    <tr>
      <td>6. Biến dạng và hòa trộn</td>
      <td>Inverse Warping + Bilinear Interpolation + Distance-weighted Feathering</td>
      <td>Inverse Warping loại bỏ hoàn toàn hiện tượng xuất hiện lỗ trống pixel; Bilinear triệt tiêu răng cưa; Feathering khử đường viền nối và làm mượt độ chênh lệch phơi sáng tự nhiên mà không tốn nhiều bộ nhớ như Multi-band</td>
      <td>Mô hình tính toán dạng Gather (1 thread = 1 pixel đầu ra) loại trừ hoàn toàn race condition; Bilinear được hỗ trợ trực tiếp bằng phần cứng Texture Memory của GPU; dễ gộp nhân (Kernel Fusion) để giảm đọc/ghi Global Memory</td>
    </tr>
  </tbody>
</table>



### 3.1 Giai đoạn 1 – Tiền xử lý: Grayscale cố định + Separable Gaussian Blur

#### Cơ chế hoạt động

**Bước 1 – RGB → Grayscale (công thức cố định BT.601):**

```
Gray = 0.299·R + 0.587·G + 0.114·B
```

Trọng số cố định → mỗi pixel RGB được xử lý **độc lập hoàn toàn**, không có sự phụ thuộc dữ liệu giữa các pixel.

**Bước 2 – Separable Gaussian Blur (tách kernel 2D thành 2 lần nhân chập 1D):**

Tận dụng tính chất phân ly (separability) của kernel Gaussian:

```
G(x,y) = g(x) · g(y)  →  Pass 1: nhân chập 1D theo hàng
                         Pass 2: nhân chập 1D theo cột
```

Độ phức tạp giảm từ **O(k²·N)** xuống **O(2k·N)** với N là tổng số pixel và k là kích thước kernel.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| σ | Độ lệch chuẩn Gaussian, kiểm soát mức độ làm mịn | 1.0 – 1.6 |
| k | Kích thước kernel, k = 2⌈3σ⌉+1 | 7 (với σ = 1.0) |

#### Lý do phù hợp với song song hóa

- **Grayscale:** Mỗi pixel độc lập → map trực tiếp `1 thread = 1 pixel`, không cần đồng bộ.
- **Pass 1D ngang:** Mỗi hàng (row) độc lập → song song hóa theo hàng.
- **Pass 1D dọc:** Mỗi cột (column) độc lập → tương tự.
- **Cache-friendly:** Đọc bộ nhớ tuần tự 1D thay vì nhảy cách 2D → giảm cache miss đáng kể.


### 3.2 Giai đoạn 2 – Harris Corner + NMS + Grid-based Top-K

Nhóm chốt **Harris Corner** làm detector duy nhất cho V0–V3 để bảo đảm phép đo speedup so sánh cùng thuật toán và tham số. 

#### Cơ chế hoạt động

- **Bước 1 – Tính gradient:** Dùng Sobel $3 \times 3$ để tính mức biến thiên cường độ theo hai hướng $I_x$ và $I_y$ tại mỗi pixel.
- **Bước 2 – Xây dựng structure tensor:** Tính ba tích $I_x^2$, $I_y^2$, $I_xI_y$, sau đó làm mượt từng tích bằng Gaussian $G_{\sigma_w}$:
  $$S_{xx} = G_{\sigma_w} \ast I_x^2, \qquad S_{yy} = G_{\sigma_w} \ast I_y^2, \qquad S_{xy} = G_{\sigma_w} \ast (I_xI_y)$$
  Structure tensor tại mỗi pixel là:
  $$M = \begin{bmatrix} S_{xx} & S_{xy} \\ S_{xy} & S_{yy} \end{bmatrix}$$
- **Bước 3 – Tính Harris response:** Từ structure tensor $M$, tính:
  $$R = \det(M) - k\,\operatorname{trace}(M)^2 = S_{xx}S_{yy} - S_{xy}^2 - k(S_{xx} + S_{yy})^2$$
  $R$ lớn và dương biểu thị ứng viên góc; $R$ âm thường là cạnh; $R$ gần 0 là vùng phẳng.
- **Bước 4 – Lọc và chọn keypoint:** Với bán kính NMS $r_{\mathrm{NMS}}$ và biên keypoint cấu hình $b_{\mathrm{kp}}$, code dùng $b = \max(b_{\mathrm{kp}}, r_{\mathrm{NMS}})$ và miền ảnh hợp lệ:
  $$\Omega = \{(x,y) \mid b \le x < W-b,\; b \le y < H-b\}$$
  Response lớn nhất và ngưỡng tương đối được tính bởi:
  $$R_{\max} = \max_{(u,v) \in \Omega} R(u,v), \qquad R(x,y) > \alpha R_{\max}$$
  Sau đó áp dụng NMS để chỉ giữ cực đại cục bộ; các response bằng nhau được phân xử theo thứ tự raster. Cuối cùng, Grid-based Top-K giữ tối đa $K$ điểm trong mỗi cell theo thứ tự tăng dần của bộ $(-R, y, x)$, tức là ưu tiên response cao hơn rồi đến tọa độ theo thứ tự raster.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị V0 hiện tại |
|---------|---------|------------------|
| `k` | Độ nhạy của Harris response với cạnh | 0.04 |
| `σ_w` | Độ lệch chuẩn Gaussian khi làm mượt structure tensor | 1.0 |
| Tensor window | Kích thước cửa sổ làm mượt tensor | 5×5 |
| `α` | Hệ số threshold tương đối trong `R > α·max(R)` | 0.01 |
| NMS window | Cửa sổ giữ cực đại cục bộ | 3×3 |
| Grid cell | Kích thước ô dùng cho Spatial Top-K | 32×32 pixel |
| K/cell | Số keypoint tối đa giữ lại trong mỗi ô | 5 |
| Keypoint border | Biên loại keypoint, đồng thời chừa vùng cho descriptor Stage 3 | 16 pixel |

#### Lý do phù hợp với song song hóa

- **Sobel, tensor và Harris response:** Mỗi pixel được ánh xạ trực tiếp thành `1 thread = 1 pixel`; luồng điều khiển đều, ít phân kỳ warp.
- **Stencil cục bộ:** Các pixel lân cận được tái sử dụng bằng shared-memory tiling có halo, giúp giảm số lần đọc global memory.
- **Threshold và NMS:** Mỗi thread kiểm tra response của pixel mình trên response map đã hoàn tất, không có phụ thuộc ghi giữa các pixel.
- **Tìm `max(R)` và thu thập keypoint:** Có thể dùng các mẫu song song chuẩn như reduction, prefix-sum compaction hoặc warp-level aggregation.
- **Grid-based Top-K:** Mỗi ô ảnh có thể được một CUDA block xử lý độc lập, vừa song song hóa chọn điểm vừa bảo đảm phân bố không gian.
- **Khả năng tối ưu theo phiên bản:** V1 dùng global memory; V2 dùng shared-memory tiling; V3 tối ưu compaction và Spatial Top-K dựa trên bottleneck đo được.

Bảng trên ghi cấu hình V0 hiện đang được triển khai. Khi xây dựng V1–V3, nhóm giữ nguyên các giá trị và output semantics này để phép đo speedup so sánh cùng thuật toán.


### 3.3 Giai đoạn 3 – Mô tả điểm đặc trưng: ORB descriptor (BRIEF có xoay, 256 bit)

Nhóm chọn **ORB descriptor** (BRIEF có xoay theo hướng keypoint) để mô tả vùng lân cận của mỗi keypoint bằng một chuỗi nhị phân 256 bit.

#### Cơ chế hoạt động

- **Bước 1 – Lấy vùng lân cận (patch):** Với mỗi keypoint từ Giai đoạn 2, xét patch kích thước $S \times S$ quanh keypoint trên ảnh xám đã làm mịn ở Giai đoạn 1. Làm mịn trước là cần thiết vì mỗi bit chỉ dựa trên so sánh hai pixel, rất dễ bị nhiễu làm sai.
- **Bước 2 – Tính hướng keypoint (Intensity Centroid):** Tính các moment của patch trong vùng tròn bán kính $r$:
  $$m_{pq} = \sum_{x,y} x^p y^q \, I(x, y)$$
  Hướng của keypoint là hướng từ tâm patch tới trọng tâm cường độ:
  $$\theta = \operatorname{atan2}(m_{01}, m_{10})$$
  Khi ảnh bị xoay, trọng tâm cường độ xoay theo, nên $\theta$ cho biết patch đang lệch bao nhiêu so với hướng chuẩn.
- **Bước 3 – Xoay mẫu so sánh (Steering):** Tập $n$ cặp điểm cố định $\{(\mathbf{x}_i, \mathbf{y}_i)\}$ quanh tâm patch được xoay theo $\theta$ bằng ma trận quay $R_\theta$:
  $$(\mathbf{x}_i^\theta, \mathbf{y}_i^\theta) = (R_\theta \mathbf{x}_i,\; R_\theta \mathbf{y}_i)$$
  Góc $\theta$ được rời rạc hóa theo bước 12°, nên có thể tính sẵn các mẫu đã xoay và chỉ cần tra bảng.
- **Bước 4 – Phép thử nhị phân (Binary Test):** Mỗi cặp điểm cho ra 1 bit:
  $$\tau(\mathbf{x}_i, \mathbf{y}_i) = \begin{cases} 1 & \text{nếu } I(\mathbf{x}_i^\theta) < I(\mathbf{y}_i^\theta) \\ 0 & \text{ngược lại} \end{cases}$$
  Ghép $n = 256$ bit lại thành descriptor 32 byte của keypoint.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `S` (patch size) | Kích thước vùng lân cận dùng để tính hướng và so sánh | 31×31 |
| `r` | Bán kính vùng tròn khi tính intensity centroid | 15 pixel |
| `n` | Số cặp điểm so sánh, tức độ dài descriptor | 256 bit |
| Bước góc | Độ rời rạc hóa hướng $\theta$ khi xoay mẫu | 12° (30 mức) |

#### Lý do phù hợp với song song hóa (GPU)

- **Mỗi keypoint độc lập (Embarrassingly Parallel):** Việc tính descriptor của một keypoint không phụ thuộc keypoint khác, mỗi thread có thể xử lý một keypoint.
- **Khối lượng việc đồng đều:** Mọi keypoint đều tính moment trên cùng một vùng tròn và thực hiện đúng 256 phép so sánh, nên các thread trong warp đi cùng một luồng điều khiển, ít phân kỳ.
- **Mẫu cố định trong Constant Memory:** Tập cặp điểm (kể cả các bản đã xoay) là hằng số, dùng chung cho mọi keypoint. Khi các thread trong warp cùng đọc một cặp điểm, Constant Memory phát (broadcast) giá trị cho cả warp trong một lần truy cập.
- **Đầu ra nhị phân gọn:** Mỗi descriptor chỉ 32 byte (so với 512 byte của SIFT), giảm lượng dữ liệu phải đọc ở Giai đoạn 4 và cho phép so khớp chỉ bằng phép XOR và đếm bit.

### 3.4 Giai đoạn 4 – Ghép cặp điểm: Brute-force Hamming + Lowe Ratio Test + Cross-check

Nhóm chọn **so khớp vét cạn (Brute-force) theo khoảng cách Hamming**, kết hợp hai bộ lọc **Lowe ratio test** và **cross-check** để loại các cặp ghép sai trước khi đưa vào RANSAC.

#### Cơ chế hoạt động

- **Bước 1 – Khoảng cách Hamming:** Với hai descriptor nhị phân $a, b$, khoảng cách là số bit khác nhau:
  $$d_H(a, b) = \operatorname{popcount}(a \oplus b), \quad 0 \le d_H \le 256$$
- **Bước 2 – Tìm láng giềng gần nhất (Brute-force):** Với mỗi descriptor $a_i$ của ảnh A, tính khoảng cách tới **mọi** descriptor $b_j$ của ảnh B, giữ lại láng giềng gần nhất $j^*$ cùng hai khoảng cách nhỏ nhất:
  $$d_1 = d_H(a_i, b_{j^*}), \qquad d_2 = \min_{j \ne j^*} d_H(a_i, b_j)$$
- **Bước 3 – Lowe ratio test:** Chỉ chấp nhận cặp $(i, j^*)$ nếu láng giềng gần nhất tốt hơn hẳn láng giềng thứ hai:
  $$\frac{d_1}{d_2} < \rho$$
  Ở vùng có hoa văn lặp lại (cửa sổ, gạch lát), $d_1 \approx d_2$ nên cặp bị loại vì không đủ phân biệt.
- **Bước 4 – Cross-check (Mutual Nearest Neighbor):** Cặp $(i, j^*)$ chỉ được giữ nếu $a_i$ cũng là láng giềng gần nhất của $b_{j^*}$ khi tìm theo chiều ngược lại B → A. Bước này loại các trường hợp nhiều điểm của A cùng trỏ vào một điểm của B.

Kết quả là tập cặp điểm tương ứng $\{(\mathbf{p}_i, \mathbf{p}'_{j^*})\}$ làm đầu vào cho Giai đoạn 5.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `ρ` (ratio) | Ngưỡng của Lowe ratio test; càng nhỏ càng chặt | 0.7 – 0.8 |
| Cross-check | Bật/tắt kiểm tra láng giềng gần nhất hai chiều | Bật |

#### Lý do phù hợp với song song hóa (GPU)

- **Bài toán all-pairs giống nhân ma trận:** Cần tính $N_A \times N_B$ khoảng cách độc lập nhau; mỗi thread có thể xử lý một descriptor của A và duyệt toàn bộ B.
- **Phép tính khoảng cách rất rẻ:** Mỗi khoảng cách chỉ gồm XOR và đếm bit 1 trên 256 bit; GPU hỗ trợ đếm bit bằng phần cứng nên phép tính này rất nhanh.
- **Tái sử dụng dữ liệu cao, hợp với Shared-memory Tiling:** Mỗi descriptor của B được mọi thread đọc lại. Nạp từng khối (tile) descriptor B vào Shared Memory để cả block dùng chung sẽ giảm mạnh số lần đọc Global Memory, tương tự tiled matrix multiplication.
- **Tìm $d_1$, $d_2$ là bài toán Reduction:** Khi chia việc duyệt B cho nhiều thread, kết quả cục bộ được gộp bằng warp shuffle; cross-check có cùng cấu trúc, chỉ đổi vai trò A và B.
- **Không dùng KD-tree / FLANN:** Các cấu trúc cây khiến mỗi thread duyệt một nhánh khác nhau (phân kỳ warp, truy cập bộ nhớ không đều) và chỉ cho kết quả xấp xỉ; brute-force cho kết quả chính xác với luồng thực thi đồng nhất.

### 3.5 Giai đoạn 5 – Ước lượng Homography: RANSAC + 4-point DLT + Inlier Refinement

Nhóm chọn **RANSAC chuẩn kết hợp DLT 4 điểm và bước tinh chỉnh (Refinement)** làm giải thuật ước lượng ma trận biến đổi xạ ảnh $H_{3 \times 3}$.

#### Cơ chế hoạt động

- **Bước 1 – Lấy mẫu ngẫu nhiên tối thiểu:** Chọn ngẫu nhiên 4 cặp điểm tương ứng không đồng tuyến từ tập kết quả so khớp ở Giai đoạn 4.
- **Bước 2 – Giải ma trận Homography giả thuyết ($H$):** Thiết lập hệ phương trình tuyến tính $Ah = 0$ ($A$ kích thước $8 \times 9$, mỗi cặp điểm cho 2 phương trình) và giải qua phép phân tích kỳ dị (SVD) hoặc khử Gauss để tìm ma trận $H$ kích thước $3 \times 3$.
- **Bước 3 – Đánh giá giả thuyết và đếm Inliers:** Chiếu toàn bộ các điểm đặc trưng của ảnh nguồn sang ảnh đích qua $H$, tính sai số hình học (reprojection error):
  $$d(p_i', H p_i) = \sqrt{(x_i' - \hat{x}_i)^2 + (y_i' - \hat{y}_i)^2}$$
  Điểm được tính là inlier nếu sai số $d < \tau$ (ngưỡng khoảng cách).
- **Bước 4 – Cập nhật ma trận tối ưu:** Lặp lại quá trình trên $N$ lần; lưu lại ma trận $H$ sở hữu số lượng inliers lớn nhất.
- **Bước 5 – Tinh chỉnh (Refinement):** Giải lại DLT tổng quát bằng phương pháp bình phương tối thiểu (least squares) trên toàn bộ tập inliers tối ưu để triệt tiêu sai số cục bộ.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `N_iter` | Số vòng lặp lấy mẫu giả thuyết RANSAC | 1000 – 2000 |
| `τ` (threshold) | Ngưỡng khoảng cách sai số reprojection để chấp nhận là Inlier | 3.0 – 5.0 pixel |
| `inlier_ratio_target` | Tỉ lệ inliers kỳ vọng để dừng sớm (early stopping) | 0.7 – 0.8 |

#### Lý do phù hợp với song song hóa (GPU)

- **Các giả thuyết độc lập (Embarrassingly Parallel):** Mỗi vòng lặp sinh và kiểm tra một giả thuyết $H$ hoàn toàn độc lập với nhau, cho phép phân bổ trực tiếp cho các CUDA Block hoặc Warp chạy song song đồng thời.
- **Không gây phân kỳ luồng (No Warp Divergence):** Khác với PROSAC (không gian mẫu phụ thuộc trạng thái động tăng dần), RANSAC chuẩn có luồng thực thi đồng nhất cho mọi giả thuyết, tận dụng tối đa năng lực xử lý của warp.
- **Đếm Inlier bằng Parallel Reduction:** Bước kiểm tra sai số và đếm số điểm inlier của $M$ cặp điểm được ánh xạ thành bài toán song song hóa reduction trên Shared Memory cực kỳ hiệu quả.


### 3.6 Giai đoạn 6 – Biến dạng và hòa trộn: Inverse Warping + Bilinear Interpolation + Distance-weighted Feathering

Nhóm chọn giải pháp **Inverse Warping kết hợp lấy mẫu song tuyến tính (Bilinear Interpolation)** và kỹ thuật hòa trộn **Distance-weighted Feathering** để tạo ảnh toàn cảnh cuối cùng.

#### Cơ chế hoạt động

- **Bước 1 – Xác định kích thước Canvas và Ma trận tịnh tiến ($T$):**
  - Chiếu 4 góc của ảnh nguồn (ảnh B) sang hệ tọa độ của ảnh tham chiếu (ảnh A) qua ma trận $H_{2 \to 1}$.
  - Xác định bounding box bao phủ toàn bộ 8 góc của hai ảnh: $[x_{min}, x_{max}] \times [y_{min}, y_{max}]$.
  - Kích thước canvas toàn cảnh được tính: $W_{pano} = x_{max} - x_{min}$, $H_{pano} = y_{max} - y_{min}$.
  - Thiết lập ma trận tịnh tiến $T$ với độ dịch chuyển $(t_x, t_y) = (-x_{min}, -y_{min})$ để đưa toàn bộ tọa độ về phần dương của canvas.

- **Bước 2 – Chiếu ngược tọa độ (Inverse Warping):**
  - Với mỗi vị trí nguyên $(x, y)$ trên canvas đích, tính tọa độ thực $(u, v)$ tương ứng trên ảnh gốc thông qua ma trận nghịch đảo:
    $$\begin{bmatrix} u' \\ v' \\ w' \end{bmatrix} = (T \cdot H)^{-1} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} \implies u = \frac{u'}{w'}, \quad v = \frac{v'}{w'}$$
  - Cơ chế *Inverse Warping* đảm bảo mỗi pixel trên canvas đích luôn tìm được giá trị màu tương ứng, khắc phục triệt để hiện tượng xuất hiện các lỗ trống điểm ảnh (holes) của phương pháp chiếu xuôi (*Forward Warping*).

- **Bước 3 – Lấy mẫu nội suy song tuyến tính (Bilinear Interpolation):**
  - Do tọa độ $(u, v)$ là số thực, giá trị màu được nội suy từ 4 pixel nguyên lân cận bao quanh:
    $$I(u, v) = (1 - \Delta u)(1 - \Delta v)I(u_0, v_0) + \Delta u(1 - \Delta v)I(u_1, v_0) + (1 - \Delta u)\Delta v I(u_0, v_1) + \Delta u \Delta v I(u_1, v_1)$$
    trong đó $\Delta u = u - u_0, \Delta v = v - v_0$. Phép nội suy này giúp ảnh uốn mịn màng và loại bỏ hoàn toàn răng cưa do phép biến đổi phối cảnh gây ra.

- **Bước 4 – Hòa trộn trọng số khoảng cách (Distance-weighted Feathering):**
  - Trong vùng chồng lấn (overlap), tính khoảng cách Euclidean $d_1(x, y)$ và $d_2(x, y)$ từ pixel hiện tại tới đường biên gần nhất của từng ảnh thành phần.
  - Trộn màu tuyến tính theo trọng số khoảng cách:
    $$I_{blend}(x, y) = \frac{d_1(x, y) \cdot I_1(x, y) + d_2(x, y) \cdot I_2(x, y)}{d_1(x, y) + d_2(x, y)}$$
  - Khi tiến gần viền của ảnh 1 thì $d_1 \to 0$, trọng số của ảnh 1 giảm dần về 0 và chuyển tiếp mượt mà sang ảnh 2, giúp xóa nhòa đường nối cắt và cân bằng độ chênh lệch ánh sáng giữa hai góc chụp.

#### Đặc tính tham số: Phương pháp phi tham số (Parameter-free)

Một ưu điểm lớn của sự kết hợp **Inverse Warping + Distance-weighted Feathering** là phương pháp này **hoàn toàn không có tham số cần tinh chỉnh (Parameter-free)**:
- **Tính tất định trong uốn ảnh:** Tọa độ lấy mẫu của từng pixel đầu ra được xác định chính xác và duy nhất thông qua ma trận biến đổi xạ ảnh $H_{2 \to 1}$ đã tìm được ở Giai đoạn 5.
- **Tự động thích ứng trong hòa trộn:** Trọng số trộn màu $\alpha_1, \alpha_2$ tại mỗi điểm được tính toán tự động dựa trên khoảng cách hình học thực tế đến viền ảnh ($d_1, d_2$). Không cần đặt ngưỡng cố định hay bề rộng dải cắt cứng nhắc, giúp pipeline hoạt động ổn định trên mọi cặp ảnh có góc xoay và độ chồng lấp khác nhau.

#### Lý do phù hợp với song song hóa (GPU)

- **Mô hình tính toán dạng Gather (1 thread = 1 pixel output):** Toàn bộ không gian canvas đích được phân rã thành lưới luồng 2D (`1 thread = 1 pixel`). Mỗi luồng chỉ đọc từ ảnh nguồn và ghi đúng vào một vị trí pixel đích, loại bỏ hoàn toàn xung đột ghi bộ nhớ (*race condition*) mà không cần bất kỳ lệnh khóa hay đồng bộ nào.
- **Tận dụng phần cứng Texture Memory:** GPU trang bị bộ điều khiển Texture chuyên dụng có khả năng nội suy Bilinear trực tiếp ở cấp độ phần cứng chỉ trong 1 chu kỳ truy cập, đồng thời sở hữu bộ đệm không gian 2D (Spatial Cache) giúp tăng tốc độ lấy mẫu khi ảnh bị biến dạng hình học.
- **Khả năng gộp nhân (Kernel Fusion):** Toàn bộ quy trình từ tính tọa độ nghịch đảo, lấy mẫu Bilinear cho đến hòa trộn Feathering có thể gộp chung vào duy nhất một CUDA kernel. Nhờ đó, chương trình không phải lưu các bức ảnh uốn trung gian ra bộ nhớ toàn cục (Global Memory), tiết kiệm đáng kể băng thông VRAM.
- **Tiết kiệm tài nguyên so với Multi-band Blending:** Kỹ thuật Feathering có chi phí tính toán nhẹ và độ phức tạp thấp hơn nhiều so với việc phân rã tháp Laplacian nhiều tầng của Multi-band, đảm bảo khả năng mở rộng tốc độ (speedup) tốt trên các tập ảnh phân giải lớn (Full HD, 4K).


---

## 4. Phiên bản tuần tự (V0)

Phiên bản V0 là cài đặt C/C++ **đơn luồng** (không dùng GPU) của toàn bộ pipeline ghép ảnh.

---

## 5. Kết quả và phân tích hiệu năng V0

---

### 5.1 Độ chính xác

### 5.2 Ảnh toàn cảnh đầu ra

### 5.3 Bảng thời gian thực thi theo từng giai đoạn

### 5.4 Phân tích giai đoạn chiếm nhiều thời gian nhất

### 5.5 Ước lượng theo Định luật Amdahl

---

## 6. Bộ dữ liệu

Bộ dữ liệu do nhóm tự chụp bằng điện thoại (iPhone, định dạng gốc `.HEIC`), gồm **9 cặp ảnh** chia thành hai nhóm:

- **Standard pairs (7 cặp):** 3 cặp trong nhà (Indoor) và 4 cặp ngoài trời (Outdoor). Đây là các cặp đủ sáng, có vùng chồng lấp rõ, dùng để kiểm tra tính đúng đắn và đo hiệu năng chính.
- **Challenge pairs (2 cặp):** các cặp chụp ban đêm hoặc thiếu sáng, dùng để kiểm tra độ bền của pipeline trong điều kiện khó.

### 6.1 Quy chuẩn ảnh và cách chọn cặp ảnh

**Quy trình chuẩn hóa:**

1. **Chuyển định dạng:** Ảnh gốc `.HEIC` được chuyển sang `.jpg`. Pillow không đọc được HEIC nếu không có plugin (ví dụ `pillow-heif`), và V0 đọc ảnh bằng `stb_image`, vốn không hỗ trợ HEIC.
2. **Cắt tỉ lệ 16:9 ở giữa (center-crop):** Công cụ `tools/resize_images.py` cắt bỏ phần thừa ở hai bên (với ảnh rộng hơn 16:9) hoặc ở trên và dưới (với ảnh 4:3 hoặc ảnh dọc) để ảnh không bị méo.
3. **Resize về 3 độ phân giải** bằng nội suy Lanczos:

| Tên | Độ phân giải | Số pixel | Mục đích                          |
|-----|--------------|----------|-----------------------------------|
| Nhỏ | 1280 × 720   | ≈ 0.92 MP | Phát triển nhanh và debug         |
| Vừa | 1920 × 1080  | ≈ 2.07 MP | Benchmark chính                   |
| Lớn | 3840 × 2160  | ≈ 8.29 MP | Kiểm tra khả năng mở rộng của GPU |

Ba mức có số pixel tăng lần lượt khoảng ×2.25 và ×4, đủ để quan sát thời gian của từng giai đoạn tăng theo kích thước ảnh như thế nào (tuyến tính với Giai đoạn 1, 2, 6; phụ thuộc số keypoint với Giai đoạn 3, 4, 5).

**Tiêu chí chọn cặp ảnh:**

- **Chụp xoay tại chỗ:** Người chụp đứng yên và xoay máy theo phương ngang để giảm thị sai (parallax). Nhờ vậy, quan hệ giữa hai ảnh xấp xỉ được bằng một homography $H_{3\times3}$ (Giai đoạn 5).
- **Vùng chồng lấp khoảng 30–60%:** Đủ để có nhiều cặp điểm tương ứng nhưng vẫn mở rộng được góc nhìn.
- **Giữ nguyên phơi sáng và tỉ lệ:** Chụp liên tiếp hai ảnh, không zoom; điều này phù hợp với giả định của ORB descriptor là gần như không đổi tỉ lệ (Giai đoạn 3).
- **Có đủ chi tiết trong vùng chồng lấp:** Ưu tiên khung cảnh có góc cạnh (tòa nhà, khung cửa, biển báo) để Harris tìm được keypoint. Các vùng phẳng (bầu trời, tường trơn) được giữ lại ở mức vừa phải để kiểm tra khả năng chịu lỗi.
- **Ảnh thách thức:** Được chọn có chủ đích với các yếu tố gây khó cho ghép ảnh, gồm thiếu sáng, nhiễu cảm biến, nguồn sáng mạnh cục bộ và hoa văn lặp lại.

### 6.2 Danh sách cặp ảnh

| STT | Nhóm     | Bối cảnh | Cặp | Ảnh (A, B)           | Mô tả cặp ảnh | Đặc điểm và thách thức | Đường dẫn |
|:---:|----------|----------|:---:|----------------------|---------------|------------------------|-----------|
| 1   | Standard | Indoor   | 1   | IMG_1147, IMG_1148   | Hầm gửi xe: dãy xe đạp và xe máy, cột bê tông có bảng chữ đỏ "Phòng cháy và chữa cháy…" | Bảng chữ và đường ống trên trần cho nhiều góc tốt; dãy xe đạp có hoa văn lặp; đèn huỳnh quang gây vùng chói | [Pair 1](https://drive.google.com/drive/folders/1bazKObMIJAp9hsx5MDNH-rlytnY9nwai) |
| 2   | Standard | Indoor   | 2   | IMG_1144, IMG_1145   | Hầm gửi xe máy: hàng xe, cột và biển số xe ở tiền cảnh | Ảnh dọc (3:4); tiền cảnh rất gần máy nên dễ có thị sai; nền sàn bê tông ít đặc trưng | [Pair 2](https://drive.google.com/drive/folders/16gO73__pc9fjuR-yP3WQRpF2z6KZrdqN) |
| 3   | Standard | Indoor   | 3   | IMG_1154, IMG_1155   | Sảnh trong nhà có lưới thể thao và băng rôn "Nốt nhạc xanh", trần lắp đèn và quạt | Ảnh dọc (9:16); sàn gạch, đèn trần lặp đều đặn; băng rôn nhiều màu là vùng giàu đặc trưng | [Pair 3](https://drive.google.com/drive/folders/1WG62R6lKcFQuwD4j_w4q43EVcYCOvTP7) |
| 4   | Standard | Outdoor  | 1   | IMG_1107, IMG_1108   | Khuôn viên Trường ĐH Khoa học Xã hội và Nhân văn: tòa nhà, mái che, cây xanh, biển tên trường | Ánh sáng ban ngày tốt; tòa nhà nhiều cửa sổ cho nhiều góc; nửa trên ảnh là bầu trời ít đặc trưng | [Pair 1](https://drive.google.com/drive/folders/1Rm0PmaibfbrIco7ZQ5fKQsaHR_vvna1I) |
| 5   | Standard | Outdoor  | 2   | IMG_1109, IMG_1110   | Công trình đang xây phủ lưới xanh, rừng cây và mặt hồ | Vùng chồng lấp hẹp, chủ yếu là hàng cây; bầu trời và tán cây ít góc rõ nên dễ thiếu inlier | [Pair 2](https://drive.google.com/drive/folders/1NWiPi871r-6r_Z0oPCiRq4RiRLAnGJNq) |
| 6   | Standard | Outdoor  | 3   | IMG_1111, IMG_1112   | Đường ven công trình đang xây, hàng cây, lan can, nhà cao tầng phía xa | Ảnh A chụp ngược sáng (mặt trời trong khung) nên hai ảnh chênh lệch phơi sáng, giúp kiểm tra Feathering; cây ở tiền cảnh gây thị sai | [Pair 3](https://drive.google.com/drive/folders/1fKM8PydZAzXaC2RdpRq2jMWcoG9CpJZV) |
| 7   | Standard | Outdoor  | 4   | IMG_1129, IMG_1131   | Tòa nhà ký túc xá nhìn từ trên cao, trời nhiều mây, nhà cao tầng phía xa | Lưới cửa sổ lặp lại nhiều (dễ ghép nhầm, cần ratio test và cross-check); các vật ở xa nên thị sai thấp | [Pair 4](https://drive.google.com/drive/folders/1VwgycPRLvthP0Ei7vR9kGYw_m7chZxMw) |
| 8   | Challenge | Outdoor (đêm) | 1 | IMG_1142, IMG_1143 | Tòa nhà ký túc xá ban đêm, vài ô cửa sáng đèn, đèn đường | Thiếu sáng, nhiễu cảm biến cao; keypoint tập trung quanh vài nguồn sáng; trời đen chiếm phần lớn ảnh | [Pair 1](https://drive.google.com/drive/folders/1kzScJEqCNwaicKx45AHwkhyT-MDEookv) |
| 9   | Challenge | Outdoor (đêm) | 2 | IMG_1151, IMG_1152 | Mặt tiền tòa nhà xuống cấp ban đêm: tường bong tróc, khung kính, bạt trải sàn, dây treo | Thiếu sáng, độ tương phản thấp; khung kính lặp lại; dây mảnh và phản chiếu trên kính dễ tạo cặp ghép sai | [Pair 2](https://drive.google.com/drive/folders/1694sloEPOrGOiSdOoa9P-0gzdAGe-avD) |

**Ghi chú:**

- Hai cặp Indoor 2 và Indoor 3 được chụp dọc. Khi center-crop về 16:9, công cụ chỉ giữ một dải ngang ở giữa ảnh (khoảng 42% chiều cao với ảnh 3:4 và khoảng 32% với ảnh 9:16), nên một phần vùng chồng lấp sẽ bị mất. Nhóm giữ hai cặp này để kiểm tra pipeline khi vùng chồng lấp sau khi cắt nhỏ hơn.
- Ảnh A là ảnh tham chiếu (giữ nguyên khung), ảnh B được biến dạng vào hệ tọa độ của A ở Giai đoạn 6.

**Link bộ dữ liệu (Google Drive):** [Group 07 – Panorama Dataset](https://drive.google.com/drive/folders/1ERf5uRnQxvlTOUhPQhEolzTKGz-RPh2k)

Cấu trúc thư mục:

```
Dataset/
├── standard pairs/
│   ├── Indoor/   Pair 1 … Pair 3
│   └── Outdoor/  Pair 1 … Pair 4
└── challenge pairs/
    └── Pair 1, Pair 2
```

---

## 7. Kế hoạch GPU

Mỗi phiên bản là một phiên bản của **toàn bộ pipeline sáu giai đoạn**: 
- V0 là chuẩn C++ đơn luồng
- V1 ánh xạ các phần tính toán chính lên CUDA theo cách cơ bản
- V2 kế thừa V1 và tối ưu các nút thắt đã đo
- V3 kế thừa V2 và xử lý các nút thắt còn lại. 

Một giai đoạn có thể giữ nguyên cách cài V1 trong V2/V3 nếu số đo chưa cho thấy lợi ích của thay đổi.
Khi so sánh speedup, các phiên bản dùng cùng ảnh đầu vào, thuật toán, tham số và ngữ nghĩa đầu ra. 

### 7.1 Phạm vi chuyển lên GPU

| Giai đoạn | Phần tính toán cần ánh xạ ở V1 | Dữ liệu chuyển tiếp |
|---|---|---|
| 1. Tiền xử lý | Grayscale và hai lượt lọc Gaussian tách được | Ảnh xám đã làm mịn của A và B |
| 2. Phát hiện keypoint | Sobel, tensor Harris, response, tìm cực đại toàn ảnh, threshold, NMS và Grid Top-K | Keypoint `(x, y, score)` |
| 3. Mô tả đặc trưng | Tính hướng intensity centroid và 256 phép thử BRIEF đã xoay | Descriptor gắn đúng chỉ số keypoint |
| 4. Ghép cặp | Khoảng cách Hamming, hai láng giềng gần nhất, ratio test và cross-check | Các cặp điểm A–B |
| 5. Ước lượng homography | Tạo/đánh giá các giả thuyết RANSAC, đếm inlier và chọn giả thuyết; xử lý bước tinh chỉnh nhất quán với V0 | Ma trận H ánh xạ B → A |
| 6. Warping và blending | Kiểm tra vùng phủ, chiếu ngược trên mặt trụ, nội suy song tuyến tính và feathering cho các pixel đầu ra | Panorama |

CPU vẫn điều phối kernel, cấp phát và đọc/ghi file; thao tác I/O không tính vào thời gian compute. Dữ liệu trung gian nên ở trên GPU qua các giai đoạn khi có thể, để không phải chuyển về CPU sau mỗi stage.

### 7.2 Ánh xạ luồng cơ bản cho V1

| Giai đoạn | Đơn vị công việc CUDA V1 | Điểm cần giữ đúng với V0 |
|---|---|---|
| 1. Tiền xử lý | Một thread/pixel cho grayscale và từng lượt lọc theo hàng/cột; các lượt lọc là các kernel riêng | Hệ số Gaussian, xử lý biên và thứ tự các lượt lọc |
| 2. Phát hiện keypoint | Một thread/pixel cho Sobel, tích gradient, response và kiểm tra NMS; reduction tìm `Rmax`; gom ứng viên rồi chọn điểm theo từng ô lưới | `R > α·Rmax`, miền biên hợp lệ, hòa điểm bằng thứ tự raster, quota/cell và xếp hạng `(score giảm, y tăng, x tăng)` |
| 3. Mô tả đặc trưng | Một thread/keypoint để tính hướng và descriptor | Cùng mẫu BRIEF, lượng tử hóa góc và thứ tự keypoint–descriptor |
| 4. Ghép cặp | Một thread/descriptor truy tìm hai láng giềng gần nhất; lượt ngược phục vụ cross-check | Cùng khoảng cách Hamming, ratio test và quy tắc xử lý hòa điểm |
| 5. Ước lượng homography | Các giả thuyết RANSAC được đánh giá song song; số inlier của mỗi giả thuyết được giảm gộp | Cùng mẫu theo chỉ số giả thuyết, ngưỡng reprojection, tie-break, dừng sớm và bước tinh chỉnh |
| 6. Warping và blending | Một thread/pixel đầu ra để chiếu ngược, lấy mẫu và hòa trộn | Cùng canvas, phép chiếu mặt trụ, vùng phủ, nội suy, feathering và crop |

### 7.3 Lộ trình tối ưu V2

Bảng chỉ liệt kê các stage đã có phương án cải tiến cụ thể. Mọi phương án vẫn là **dự kiến**, chỉ triển khai khi profile của phiên bản trước xác nhận nút thắt và kết quả vẫn tương đương V0.

| Phiên bản | Giai đoạn | Cải tiến dự kiến | Điều kiện chọn và điều cần giữ đúng |
|---|---|---|---|
| V2 | 1. Tiền xử lý | Thử tile có halo cho Gaussian tách được để tái sử dụng pixel lân cận | Chỉ áp dụng nếu các lượt lọc bị giới hạn bởi truy cập bộ nhớ; giữ hệ số lọc và border policy |
| V2 | 2. Keypoint | Thử shared-memory tiling có halo cho Sobel, làm mượt tensor hoặc NMS | Chọn kernel có memory traffic là nút thắt; giữ response, threshold, NMS và tập keypoint tương đương |
| V2 | 4. Matching | Nạp từng tile descriptor để nhiều thread trong block dùng lại | Chỉ áp dụng nếu việc đọc lặp descriptor chiếm thời gian đáng kể; giữ hai láng giềng gần nhất, ratio test và cross-check |

### 7.4 Lộ trình tối ưu V3
| Phiên bản | Giai đoạn | Cải tiến dự kiến | Điều kiện chọn và điều cần giữ đúng |
|---|---|---|---|
| V3 | 2. Keypoint | Thử block/warp aggregation hoặc prefix sum để gom ứng viên; tối ưu chọn Top-K theo từng ô | Chỉ áp dụng nếu compaction/Top-K còn là nút thắt sau V2; luôn chọn theo score và cùng tie-break, không chọn theo thứ tự `atomicAdd` |

---

## Tài liệu tham khảo

Các nguồn [1]–[7] là cơ sở thuật toán; nhóm tự cài đặt và điều chỉnh các bước trong pipeline, không dùng mã triển khai của các bài báo. Riêng [8] là thư viện đọc/ghi ảnh được dùng trực tiếp. [9] là tài liệu kỹ thuật cho kế hoạch CUDA ở Mục 7.

### Thuật toán và triển khai

1. C. Harris và M. Stephens, [*A Combined Corner and Edge Detector*](https://www.bmva-archive.org.uk/bmvc/1988/avc-88-023.html), *Proceedings of the Alvey Vision Conference*, 1988. Cơ sở Harris response ở Giai đoạn 2; threshold, NMS, Grid Top-K và tham số cụ thể là thiết kế của nhóm.
2. E. Rublee, V. Rabaud, K. Konolige và G. Bradski, [*ORB: An Efficient Alternative to SIFT or SURF*](https://dev.ipol.im/~reyotero/bib/bib_all/2011_Rublee_Rabaud_ORB_alternative_to_sift_surf_willGarag_ICCV.pdf), *ICCV*, 2011. Cơ sở intensity centroid và steered BRIEF ở Giai đoạn 3; pipeline dùng Harris để phát hiện keypoint thay cho FAST của ORB gốc.
3. D. G. Lowe, [*Distinctive Image Features from Scale-Invariant Keypoints*](https://home.cis.rit.edu/~cnspci/references/dip/feature_extraction/lowe2004.pdf), *International Journal of Computer Vision*, 2004. Cơ sở phép ratio test ở Giai đoạn 4; nhóm áp dụng với khoảng cách Hamming của descriptor nhị phân.
4. M. A. Fischler và R. C. Bolles, [*Random Sample Consensus: A Paradigm for Model Fitting with Applications to Image Analysis and Automated Cartography*](https://doi.org/10.1145/358669.358692), *Communications of the ACM*, 1981. Cơ sở RANSAC ở Giai đoạn 5.
5. R. I. Hartley, [*In Defense of the Eight-Point Algorithm*](https://doi.org/10.1109/34.601246), *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 1997. Tham khảo cách chuẩn hóa tọa độ trước khi ước lượng biến đổi hình học; code của nhóm áp dụng chuẩn hóa cho bài toán homography bốn điểm.
6. M. Brown và D. G. Lowe, [*Automatic Panoramic Image Stitching using Invariant Features*](https://mattabrown.github.io/pdf/ijcv2007.pdf), *International Journal of Computer Vision*, 2007. Tham khảo hình học panorama và ý tưởng chỉnh độ nghiêng theo trục xoay ở Giai đoạn 6; phép blend trong code là feathering của nhóm.
7. M. E. O’Neill, [*PCG: A Family of Simple Fast Space-Efficient Statistically Good Algorithms for Random Number Generation*](https://www.pcg-random.org/paper.html), Harvey Mudd College Technical Report HMC-CS-2014-0905, 2014. Cơ sở hàm băm tạo mẫu cố định trong `src/common.h`, dùng cho pattern descriptor và RANSAC.
8. S. Barrett và các cộng tác viên, [*stb single-file public domain libraries*](https://github.com/nothings/stb), mã nguồn `stb_image.h` và `stb_image_write.h`. Thư viện bên ngoài được đưa vào `src/` để đọc/ghi ảnh; các phép toán ghép ảnh do nhóm cài đặt.
9. NVIDIA, [*CUDA Programming Guide*](https://docs.nvidia.com/cuda/cuda-programming-guide/), tài liệu chính thức về mô hình kernel, bộ nhớ, streams và CUDA events, dùng cho kế hoạch V1–V3 và phương pháp đo GPU.